# Section 6.1: comparison with reference posteriors

Four simulation designs, 100 replicates per setting (1600 data sets): linear designs ($n = 50$) against
spike-and-slab regression, Friedman designs ($n = 100$) against DART; $p = 100$, equicorrelated or
Toeplitz covariance with $\rho \in \{0.3, 0.7\}$, SNR $\in \{1, 5\}$.

Runtime: a GPU (A100 or L4). The network takes minutes; spike-and-slab and DART run for a few hours on
the CPU cores. Every run writes one line per data set and skips the data sets already done, so if the
session drops, rerun cell 1 and the cell you were at.

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, glob, json, shutil, subprocess, collections
# folder of this repository on your Google Drive (upload it there, or clone it with git)
CODE = '/content/drive/MyDrive/pfn-bayesian-variable-selection'
assert os.path.exists(f'{CODE}/bench/run.py'), f'repository not found in {CODE}'
os.chdir(CODE)
os.makedirs('results/logs', exist_ok=True)
!pip -q install tabulate
!nvidia-smi --query-gpu=name --format=csv,noheader
NCPU = os.cpu_count(); print('CPU cores:', NCPU)
ENV = {**os.environ, 'OMP_NUM_THREADS': '1', 'MKL_NUM_THREADS': '1', 'OPENBLAS_NUM_THREADS': '1'}

def launch(args, log):
    """Start `python -m <args>` in the background, with its log in results/logs."""
    return subprocess.Popen(['python', '-W', 'ignore', '-m'] + args, env=ENV, start_new_session=True,
                            stdout=open(f'results/logs/{log}.log', 'a'), stderr=subprocess.STDOUT)

def count(folder):
    c, e = collections.Counter(), collections.Counter()
    for f in glob.glob(f'{folder}/*.jsonl'):
        for line in open(f):
            if line.strip():
                r = json.loads(line)
                (e if r.get('error') else c)[r['method']] += 1
    return c, e

## 2. The trained network

Trains the network of the thesis (about 4 hours on an A100) unless `runs/small/final.pt` already exists. Training resumes from the last checkpoint if restarted.

In [ ]:
if os.path.exists('runs/small/final.pt'):
    print('runs/small/final.pt already there')
else:
    train = launch(['pfn.train', '--preset', 'small', '--out', 'runs/small'], 'train_small')
    print('training started; follow it with: !tail -n 2 runs/small/log.csv')

## 3. R and the BART package (for DART)

In [ ]:
if shutil.which('Rscript') is None:
    !apt-get -qq install -y r-base > /dev/null
!Rscript -e 'if (!requireNamespace("BART", quietly=TRUE)) install.packages("BART", repos="https://cloud.r-project.org", quiet=TRUE); library(BART); cat("BART ok\n")'

## 4. Run the three methods (in the background)

In [ ]:
SP = 'runs/small/final.pt'
assert os.path.exists(SP), 'the network is not trained yet'
procs = [launch(['bench.run', '--study', 'simple', '--reps', '100', '--methods', 'selectpfn',
                 '--sp', f'small={SP}', '--out', 'results/simple'], 'simple_network')]
K = max(1, NCPU - 1)
for i in range(K):
    procs.append(launch(['bench.run', '--study', 'simple', '--reps', '100', '--methods', 'spikeslab',
                         '--designs', 'LIN_EQ,LIN_TOE', '--out', 'results/simple_mcmc', '--device', 'cpu',
                         '--shard', str(i), '--nshards', str(K)], f'simple_ss_{i}'))
    procs.append(launch(['bench.run', '--study', 'simple', '--reps', '100', '--methods', 'dart',
                         '--designs', 'FRI_EQ,FRI_TOE', '--out', 'results/simple_mcmc', '--device', 'cpu',
                         '--shard', str(i), '--nshards', str(K)], f'simple_dart_{i}'))
print(len(procs), 'processes started')

## 5. Progress (rerun whenever you like)

In [ ]:
c1, e1 = count('results/simple'); c2, e2 = count('results/simple_mcmc')
for m, tot, c, e in (('selectpfn:small', 1600, c1, e1), ('spikeslab', 800, c2, e2), ('dart', 800, c2, e2)):
    print(f'{m:16s} {c[m]:5d} / {tot}   errors: {e[m]}')

## 6. Tables and Figure 6.1

Works on the data sets done so far.

In [ ]:
!python -m bench.simple_report --results results/simple,results/simple_mcmc --save results/simple_report.md
!python -m bench.make_figures results/simple_report.md results/figures
from IPython.display import Image, display
display(Image('results/figures/auprc_by_setting.png', width=800))